# Loading the Required Libraries

In [1]:
!pip install -qU bitsandbytes transformers datasets accelerate loralib einops xformers
!pip install -q -U git+https://github.com/huggingface/peft.git
!pip install rouge-score nltk datasets
!pip install evaluate

import os
import bitsandbytes as bnb
import pandas as pd
import torch
import torch.nn as nn
import transformers
from sklearn.model_selection import train_test_split
from datasets import load_dataset, Dataset
from peft import (
    LoraConfig,
    PeftConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
import evaluate
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from rouge_score import rouge_scorer
import numpy as np
import nltk
nltk.download('punkt')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 MB 25.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 106.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.4/336.4 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 95.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.4/906.4 MB 1.9 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.5 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 96.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 76.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.0 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

True

# Loading the Pre-Trained Model

In [2]:
model_id = "PrunaAI/ruslanmv-Medical-Llama3-8B-bnb-4bit-smashed"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    load_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    trust_remote_code=True,
    quantization_config=bnb_config,
)

tokenizer = AutoTokenizer.from_pretrained("ruslanmv/Medical-Llama3-8B")
tokenizer.pad_token = tokenizer.eos_token

Unused kwargs: ['load_4bit_use_double_quant']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.


config.json:   0%|          | 0.00/1.28k [00:00<?, ?B/s]

Unused kwargs: ['_load_in_4bit', '_load_in_8bit', 'quant_method']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.
/usr/local/lib/python3.10/dist-packages/transformers/quantizers/auto.py:186: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


model.safetensors.index.json:   0%|          | 0.00/89.0k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

# Preparing the Model for QLoRA

In [3]:
model = prepare_model_for_kbit_training(model)

# Configuring LoRA

In [4]:
config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, config)

# Loading and Preparing the Dataset

In [5]:
# Load the CSV file
df = pd.read_csv('/kaggle/input/frenchdataset/merged_covid_dataset_french.csv')

# df = pd.read_csv(r"D:\My Files\Career\Engineering\3\S1\NLP\Projet\Traduction Test\Data Split\Output\merged_covid_dataset_french.csv")

# Split the dataset into train and test
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# Convert DataFrames to Hugging Face datasets
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

def generate_prompt(data_point):
    return f"""
<Human>: {data_point["Question"]}
<AI>: {data_point["Answer"]}
    """.strip()

def generate_and_tokenize_prompt(data_point):
    full_prompt = generate_prompt(data_point)
    tokenized_full_prompt = tokenizer(full_prompt, padding=True, truncation=True)
    return tokenized_full_prompt

# Process the datasets
train_dataset = train_dataset.shuffle().map(generate_and_tokenize_prompt)
test_dataset = test_dataset.shuffle().map(generate_and_tokenize_prompt)

Map:   0%|          | 0/8944 [00:00<?, ? examples/s]

Map:   0%|          | 0/2237 [00:00<?, ? examples/s]

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11181 entries, 0 to 11180
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Answer    11181 non-null  object
 1   Question  11181 non-null  object
dtypes: object(2)
memory usage: 174.8+ KB


# Setting Up the Training Arguments

In [7]:
training_args = transformers.TrainingArguments(
    # auto_find_batch_size=True,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    # num_train_epochs=,
    num_train_epochs=1,
    learning_rate=2e-4,
    # bf16=True,
    save_total_limit=4,
    logging_steps=10,
    output_dir='medical-llama-covid-french-finetuned',
    save_strategy='epoch',
    evaluation_strategy="steps",  # Add evaluation evey step
    eval_steps=100,              # Evaluate every 100 steps
    warmup_ratio=0.1,           # Add warmup
    gradient_accumulation_steps=8,
    fp16=True,                  # Enable mixed precision training
    report_to="tensorboard",  # Change this to enable logging
    logging_dir="./logs",     # Add this to specify logging directory
    logging_first_step=True,  # Add this to log the first step
    logging_strategy="steps", # Add this to ensure regular logging
)

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


# Training the Model

In [8]:
trainer = transformers.Trainer(
    model=model,
    train_dataset=train_dataset,      
    eval_dataset=test_dataset,        
    args=training_args,
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

# Disable caching fo
r trainnng
model.config.use_cache = False

# Start training
trainer.train()

/usr/local/lib/python3.10/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
100,1.826900,1.788656
200,1.696200,1.730461
300,1.711200,1.694205
400,1.640300,1.666055
500,1.596600,1.638651
600,1.653500,1.614191
700,1.564000,1.596326
800,1.636100,1.579824
900,1.548800,1.566355
1000,1.540100,1.555385


TrainOutput(global_step=1118, training_loss=1.6578317953137038, metrics={'train_runtime': 39323.1361, 'train_samples_per_second': 0.227, 'train_steps_per_second': 0.028, 'total_flos': 7.4677307203584e+16, 'train_loss': 1.6578317953137038, 'epoch': 1.0})